In [ ]:
# 01_eda.ipynb
# Exploratory data analysis for both datasets:
#   Study 1  -> Polish companies bankruptcy data (UCI #365, 5 forecast horizons)
#   Study 2  -> Corporate credit ratings (OpenML #43344, 10 letter grades)
# Produces horizon / rating-distribution tables and grayscale figures (png + pdf, dpi 600).
# All code lives in this single cell; run top-to-bottom.

import os, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from scipy.io import arff
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns

# ---- paths ----
DATA = "../data"; FIG = "../results/figures"; TAB = "../results/tables"
os.makedirs(FIG, exist_ok=True); os.makedirs(TAB, exist_ok=True)

# ---- grayscale publication figure style (no titles, no captions) ----
sns.set_theme(style="white", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "figure.dpi": 110, "font.size": 11,
                     "axes.edgecolor": "black", "axes.linewidth": 0.8, "axes.grid": False})
GREYS = ["0.15", "0.45", "0.70"]
def save_fig(fig, name):
    fig.tight_layout()
    fig.savefig(f"{FIG}/{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(f"{FIG}/{name}.pdf", bbox_inches="tight")
    plt.close(fig)

# ================= Study 1: Polish bankruptcy =================
def load_polish_year(n):
    data, _ = arff.loadarff(f"{DATA}/polish/{n}year.arff")
    df = pd.DataFrame(data)
    cls = [c for c in df.columns if c.lower() == "class"][0]
    df[cls] = df[cls].apply(lambda b: int(b.decode()) if isinstance(b, bytes) else int(b))
    feats = [c for c in df.columns if c != cls]
    df.columns = [f"Attr{i+1}" for i in range(len(feats))] + ["target"]
    return df

frames = {n: load_polish_year(n) for n in range(1, 6)}
rows = []
for n in range(1, 6):
    d = frames[n]; fcols = [c for c in d.columns if c != "target"]
    rows.append({"horizon": f"{n}year", "rows": len(d), "bankrupt": int(d["target"].sum()),
                 "bankrupt_rate_pct": round(100 * d["target"].mean(), 2),
                 "missing_pct": round(100 * d[fcols].isna().mean().mean(), 2)})
polish_tab = pd.DataFrame(rows)
polish_tab.to_csv(f"{TAB}/polish_horizon_summary.csv", index=False)
print("Polish horizon summary:\n", polish_tab.to_string(index=False))
print("Pooled: rows =", sum(len(frames[n]) for n in frames),
      "| bankrupt =", sum(int(frames[n]['target'].sum()) for n in frames))

# ---- Fig 1: bankruptcy rate by forecast horizon ----
fig, ax = plt.subplots(figsize=(4.2, 3.0))
ax.bar(polish_tab["horizon"], polish_tab["bankrupt_rate_pct"], color="0.35", edgecolor="black", linewidth=0.7)
ax.set_xlabel("Forecast horizon file"); ax.set_ylabel("Bankruptcy rate (%)")
save_fig(fig, "fig01_polish_bankruptcy_rate")

# ---- Fig 2: two proxy indicators by class (5year = 1-year-to-default) ----
d5 = frames[5]
fig, axes = plt.subplots(1, 2, figsize=(6.4, 3.0))
for ax, attr, lab in zip(axes, ["Attr1", "Attr2"], ["Attr1  (net profit / total assets)", "Attr2  (total liabilities / total assets)"]):
    sub = d5[[attr, "target"]].dropna()
    lo, hi = sub[attr].quantile([0.01, 0.99])
    sub = sub[(sub[attr] >= lo) & (sub[attr] <= hi)]
    sns.boxplot(data=sub, x="target", y=attr, ax=ax, color="0.6", fliersize=1,
                linewidth=0.8, boxprops=dict(facecolor="0.8", edgecolor="black"))
    ax.set_xlabel("Bankrupt (0 = no, 1 = yes)"); ax.set_ylabel(lab)
save_fig(fig, "fig02_polish_proxy_by_class")

# ================= Study 2: corporate ratings =================
sp = pd.read_csv(f"{DATA}/corporate_rating.csv"); sp.columns = [c.strip().replace(" ", "_") for c in sp.columns]
order = ["AAA", "AA", "A", "BBB", "BB", "B", "CCC", "CC", "C", "D"]
vc = sp["Rating"].value_counts().reindex(order).fillna(0).astype(int)
band = {r: ("mid" if r in ("BBB", "BB", "B") else "extreme") for r in order}
sp_tab = pd.DataFrame({"rating": order, "count": vc.values,
                       "share_pct": (100 * vc.values / vc.values.sum()).round(2),
                       "band": [band[r] for r in order]})
sp_tab.to_csv(f"{TAB}/sp_rating_distribution.csv", index=False)
print("\nRating distribution:\n", sp_tab.to_string(index=False))
print("Mid-band share =", round(100 * sp_tab.loc[sp_tab.band == 'mid', 'count'].sum() / sp_tab['count'].sum(), 1), "%")
print("Agencies:\n", sp["Rating_Agency_Name"].value_counts().to_string())

# ---- Fig 3: rating distribution (mid band shaded darker) ----
fig, ax = plt.subplots(figsize=(5.2, 3.0))
colors = ["0.3" if band[r] == "mid" else "0.7" for r in order]
ax.bar(order, vc.values, color=colors, edgecolor="black", linewidth=0.7)
ax.set_xlabel("S&P-style letter grade"); ax.set_ylabel("Number of firm-ratings")
save_fig(fig, "fig03_sp_rating_distribution")

print("\n[01_eda] done. Figures + tables written to ../results/.")
